# V7_B_N01 — Outbreak Signals Before Alarm

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft using synthetic data. Outputs support authorized human review; they do not constitute official declarations or automated decisions.

## Decision contract
Detect unusual syndromic counts for verification and possible public-health response. Owner: public-health institute/Ministry of Health. A signal is not a confirmed outbreak and must not identify patients publicly.

In [1]:
import numpy as np,pandas as pd
rng=np.random.default_rng(7101); weeks=np.arange(1,105); base=18+4*np.sin(2*np.pi*weeks/52); counts=rng.poisson(base); counts[96:100]+=np.array([7,13,18,10]); df=pd.DataFrame({'week':weeks,'cases':counts,'reports_expected':20,'reports_received':rng.integers(16,21,len(weeks))}); df.tail(12)

## Surveillance quality
Counts depend on reporting completeness, case definitions, care seeking, testing, and duplication. Low completeness triggers data verification before escalation.

In [2]:
df['completeness']=df.reports_received/df.reports_expected; df['quality_pass']=df.completeness>=.9; print(df.tail(12)[['week','cases','completeness','quality_pass']].to_string(index=False))

 week  cases  completeness  quality_pass
   93     12          1.00          True
   94     13          0.80         False
   95     15          0.95          True
   96     18          1.00          True
   97     22          1.00          True
   98     26          0.90          True
   99     32          0.90          True
  100     19          0.80         False
  101     21          0.85         False
  102     11          1.00          True
  103     17          0.90          True
  104     15          1.00          True


## Historical seasonal baseline
Use prior comparable weeks and an over-dispersion allowance. Operational thresholds require epidemiological validation and prospective performance review.

In [3]:
df['season_week']=(df.week-1)%52+1; hist=df[df.week<=52].groupby('season_week').cases.agg(['mean','std']).reindex(range(1,53)); hist['std']=hist['std'].fillna(np.sqrt(hist['mean'].clip(lower=1))); df['expected']=df.season_week.map(hist['mean']); df['upper']=df.expected+2.5*df.season_week.map(hist['std']); df['raw_signal']=df.cases>df.upper; print(df.tail(12)[['week','cases','expected','upper','raw_signal']].round(1).to_string(index=False))

 week  cases  expected  upper  raw_signal
   93     12      16.0   26.0       False
   94     13      19.0   29.9       False
   95     15      15.0   24.7       False
   96     18       8.0   15.1        True
   97     22      12.0   20.7        True
   98     26      10.0   17.9        True
   99     32      14.0   23.4        True
  100     19      12.0   20.7       False
  101     21      15.0   24.7       False
  102     11      18.0   28.6       False
  103     17      22.0   33.7       False
  104     15      14.0   23.4       False


## Consecutive-signal and quality gate
One unusual week may be noise. Consecutive signals increase concern, while incomplete reporting causes abstention.

In [4]:
df['consecutive']=df.raw_signal & df.raw_signal.shift(1,fill_value=False); df['disposition']=np.select([~df.quality_pass,df.consecutive,df.raw_signal],['ABSTAIN—VERIFY REPORTING','ESCALATE—EPIDEMIOLOGICAL REVIEW','VERIFY SIGNAL'],default='ROUTINE'); print(df.tail(12)[['week','cases','disposition']].to_string(index=False))

 week  cases                     disposition
   93     12                         ROUTINE
   94     13        ABSTAIN—VERIFY REPORTING
   95     15                         ROUTINE
   96     18                   VERIFY SIGNAL
   97     22 ESCALATE—EPIDEMIOLOGICAL REVIEW
   98     26 ESCALATE—EPIDEMIOLOGICAL REVIEW
   99     32 ESCALATE—EPIDEMIOLOGICAL REVIEW
  100     19        ABSTAIN—VERIFY REPORTING
  101     21        ABSTAIN—VERIFY REPORTING
  102     11                         ROUTINE
  103     17                         ROUTINE
  104     15                         ROUTINE


## Alert audit and proportional response
The output records reason, data quality, owner, and response deadline. Confirmation requires case investigation, laboratory/clinical evidence where appropriate, and established protocols.

In [5]:
alerts=df[df.disposition!='ROUTINE'][['week','cases','expected','upper','completeness','disposition']].copy(); alerts['owner']='Surveillance duty officer'; alerts['status']='PENDING HUMAN REVIEW'; print(alerts.tail().round(2).to_string(index=False))

 week  cases  expected  upper  completeness                     disposition                     owner               status
   97     22      12.0  20.66          1.00 ESCALATE—EPIDEMIOLOGICAL REVIEW Surveillance duty officer PENDING HUMAN REVIEW
   98     26      10.0  17.91          0.90 ESCALATE—EPIDEMIOLOGICAL REVIEW Surveillance duty officer PENDING HUMAN REVIEW
   99     32      14.0  23.35          0.90 ESCALATE—EPIDEMIOLOGICAL REVIEW Surveillance duty officer PENDING HUMAN REVIEW
  100     19      12.0  20.66          0.80        ABSTAIN—VERIFY REPORTING Surveillance duty officer PENDING HUMAN REVIEW
  101     21      15.0  24.68          0.85        ABSTAIN—VERIFY REPORTING Surveillance duty officer PENDING HUMAN REVIEW


## Error trade-offs and monitoring
Missed outbreaks and false alarms have different consequences. Track verification results, timeliness, sensitivity, positive predictive value, workload, and affected-population impacts.

In [6]:
review=pd.DataFrame({'alert':[1,2,3,4,5,6],'verified_event':[0,0,1,1,1,0],'response_within_sla':[1,1,1,0,1,1]}); ppv=review.verified_event.mean(); sla=review.response_within_sla.mean(); print('VERIFIED_SHARE',round(ppv,2),'SLA_SHARE',round(sla,2))

VERIFIED_SHARE 0.5 SLA_SHARE 0.83


## Exercises
1. Add a moving-baseline method. 2. Test threshold sensitivity. 3. Define a confirmation workflow. 4. Explain why public communication cannot be automated from this notebook.

## Exact solutions
1. Use only prior weeks, preserve seasonality, and avoid contaminating baselines with active events. 2. Compare verified-event sensitivity, false alerts, delay, workload, and subgroup/geographic effects. 3. Assign triage, data verification, epidemiological assessment, laboratory/clinical review, authorization, communication, closure, and retrospective review. 4. Communication carries legal, scientific, behavioural, privacy, and political consequences and requires authorized contextual judgement.

In [7]:
assert set(alerts.status)=={'PENDING HUMAN REVIEW'} and len(alerts)>0; print('V7_B_N01_COMPLETE_EXECUTION_PASS')

V7_B_N01_COMPLETE_EXECUTION_PASS
